# Table 1: Formula-Level Comparison

## Step 1: Standalone temperature-dependent sigma-star calculation

This step computes the pure creep-law, global geometry-corrected stress, and temperature-dependent geometry-corrected stress results.


In [1]:
"""
Standalone temperature-dependent sigma-star comparison.

This step does NOT use ML or cross-validation. It compares the formula-level
creep-law fits:

    1. Pure creep law using stress only
    2. Global extended sigma-star
    3. Temperature-dependent sigma-star

For each temperature, log10(tR) is fitted as a linear function of the corrected
stress. Sigma-star parameters are optimized on the full filtered dataset, so the
results are in-sample formula-fit results.
"""

import os
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution, minimize
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="Polyfit may be poorly conditioned")


DATA_PATH = os.path.join("..", "Dataset", "Creep_Properties_Dataset.xlsx")

SHAPES = ["Flat", "Cylindrical"]

COMP_FEATURES = [
    "C, wt.%", "Si, wt.%", "Mn, wt.%", "P, wt.%", "S, wt.%", "Ni, wt.%",
    "Cr, wt.%", "Mo, wt.%", "Al, wt.%", "N, wt.%", "Ti, wt.%", "Fe, wt.%",
    "Nb, wt.%", "B, wt.%", "Cu, wt.%", "Co, wt.%",
]

RPV_ALLOYS = {"SA533B", "SA508", "A508/A533"}

DATASETS = {
    "SS316 Family": {"alloys": ["SS316H", "SS316L", "SS316LN", "SS316FR"], "sigma_params": 3},
    "SS304 Family": {"alloys": ["SS304H", "SS304J", "SS304L"], "sigma_params": 3},
    "RPV Family": {"alloys": ["SA533B", "SA508", "A508/A533"], "sigma_params": 2},
    "SS316H": {"alloys": ["SS316H"], "sigma_params": 3},
    "SS304H": {"alloys": ["SS304H"], "sigma_params": 3},
    "SS316L": {"alloys": ["SS316L"], "sigma_params": 3},
    "SS304J": {"alloys": ["SS304J"], "sigma_params": 3},
    "SS316FR": {"alloys": ["SS316FR"], "sigma_params": 3},
    "A709": {"alloys": ["A709"], "sigma_params": 3},
    "IN-617": {"alloys": ["IN-617"], "sigma_params": 3},
    "SS316LN": {"alloys": ["SS316LN"], "sigma_params": 3},
}


def impute_rpv_cross_section(df: pd.DataFrame) -> pd.DataFrame:
    """RPV Cross-Sectional Area"""
    df = df.copy()
    rpv_mask = df["Alloy Designation"].isin(RPV_ALLOYS)
    missing = rpv_mask & df["Cross-Sectional Area"].isna()

    cyl = missing & (df["Specimen Shape"] == "Cylindrical")
    d = df.loc[cyl, "Thickness (mm)"]
    df.loc[cyl, "Cross-Sectional Area"] = np.pi * (d / 2) ** 2
    n_cyl = int(cyl.sum())

    flat = missing & (df["Specimen Shape"] == "Flat")
    rpv_median_overall = df.loc[
        rpv_mask & df["Cross-Sectional Area"].notna(), "Cross-Sectional Area"
    ].median()
    n_flat = 0
    for alloy in RPV_ALLOYS:
        alloy_flat = flat & (df["Alloy Designation"] == alloy)
        if not alloy_flat.any():
            continue
        alloy_median = df.loc[
            rpv_mask
            & (df["Alloy Designation"] == alloy)
            & df["Cross-Sectional Area"].notna(),
            "Cross-Sectional Area",
        ].median()
        fill_val = alloy_median if not np.isnan(alloy_median) else rpv_median_overall
        df.loc[alloy_flat, "Cross-Sectional Area"] = fill_val
        n_flat += int(alloy_flat.sum())

    print(
        f"  [RPV] imputed Cross-Sectional Area: "
        f"{n_cyl} cylindrical (formula), {n_flat} flat (median)"
    )
    return df


def impute_ss316l_composition(df: pd.DataFrame) -> pd.DataFrame:
    """SS316L composition """
    df = df.copy()
    mask = df["Alloy Designation"] == "SS316L"
    comp_missing = mask & df[COMP_FEATURES[0]].isna()
    if not comp_missing.any():
        return df

    medians = df.loc[mask & ~comp_missing, COMP_FEATURES].median()
    for col in COMP_FEATURES:
        df.loc[comp_missing, col] = medians[col]
    print(
        f"  [SS316L] imputed composition for {int(comp_missing.sum())} records "
        f"using SS316L per-column medians"
    )
    return df


def metrics(y_true, y_pred):
    return (
        float(r2_score(y_true, y_pred)),
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(np.mean(np.abs(y_true - y_pred) < np.log10(2.0)) * 100.0),
    )


def creep_law_predict(log_sigma_star, log_tR, temperature):
    """Fit log10(tR) vs corrected stress separately at each temperature."""
    y_pred = np.full(len(log_tR), np.nan)
    for temp in np.unique(temperature):
        mask = temperature == temp
        if int(mask.sum()) < 3:
            continue
        if float(np.std(log_sigma_star[mask])) < 1e-12:
            y_pred[mask] = float(np.mean(log_tR[mask]))
        else:
            coeff = np.polyfit(log_sigma_star[mask], log_tR[mask], 1)
            y_pred[mask] = np.polyval(coeff, log_sigma_star[mask])

    valid = ~np.isnan(y_pred)
    if int(valid.sum()) < 2:
        return None
    return log_tR[valid], y_pred[valid]


def evaluate_corrected_stress(log_sigma_star, log_tR, temperature):
    out = creep_law_predict(log_sigma_star, log_tR, temperature)
    if out is None:
        return -np.inf, np.nan, np.nan, 0
    y_true, y_pred = out
    r2, rmse, within_2x = metrics(y_true, y_pred)
    return r2, rmse, within_2x, len(y_true)


def apply_global(params, n_params, log_sigma, log_b, log_area, log_ld):
    if n_params == 3:
        alpha, beta, gamma = params
        return log_sigma + alpha * log_b + beta * log_area + gamma * log_ld
    alpha, gamma = params
    return log_sigma + alpha * log_b + gamma * log_ld


def apply_tdependent(params, n_params, log_sigma, t_norm, log_b, log_area, log_ld):
    if n_params == 3:
        alpha0, beta0, gamma0, alpha1, beta1, gamma1 = params
        return (
            log_sigma
            + (alpha0 + alpha1 * t_norm) * log_b
            + (beta0 + beta1 * t_norm) * log_area
            + (gamma0 + gamma1 * t_norm) * log_ld
        )
    alpha0, gamma0, alpha1, gamma1 = params
    return log_sigma + (alpha0 + alpha1 * t_norm) * log_b + (gamma0 + gamma1 * t_norm) * log_ld


def objective_from_stress(log_sigma_star, log_tR, temperature):
    r2, _, _, _ = evaluate_corrected_stress(log_sigma_star, log_tR, temperature)
    if not np.isfinite(r2):
        return 1e9
    return -r2


def optimize_global(n_params, log_sigma, log_tR, temperature, log_b, log_area, log_ld):
    bounds = [(-0.5, 0.5)] * n_params
    result = differential_evolution(
        lambda p: objective_from_stress(
            apply_global(p, n_params, log_sigma, log_b, log_area, log_ld),
            log_tR,
            temperature,
        ),
        bounds=bounds,
        seed=42,
        maxiter=250,
        tol=1e-6,
        popsize=18,
        polish=True,
        workers=1,
    )
    return result.x


def optimize_tdependent(n_params, log_sigma, log_tR, temperature, t_norm, log_b, log_area, log_ld, global_params):
    if n_params == 3:
        x0 = np.array([global_params[0], global_params[1], global_params[2], 0.0, 0.0, 0.0], dtype=float)
        maxiter = 5000
    else:
        x0 = np.array([global_params[0], global_params[1], 0.0, 0.0], dtype=float)
        maxiter = 4000

    def objective(params):
        params = np.asarray(params, dtype=float)
        if np.any(params < -0.8) or np.any(params > 0.8):
            return 1e3 + float(np.sum(np.maximum(0.0, np.abs(params) - 0.8) ** 2))
        return objective_from_stress(
            apply_tdependent(params, n_params, log_sigma, t_norm, log_b, log_area, log_ld),
            log_tR,
            temperature,
        )

    result = minimize(
        objective,
        x0,
        method="Nelder-Mead",
        options={"maxiter": maxiter, "xatol": 1e-6, "fatol": 1e-8},
    )
    return result.x


def parameter_row(dataset, n_params, global_params, tdep_params):
    row = {
        "dataset": dataset,
        "sigma_params": n_params,
        "global_alpha": np.nan,
        "global_beta": np.nan,
        "global_gamma": np.nan,
        "tdep_alpha0": np.nan,
        "tdep_beta0": np.nan,
        "tdep_gamma0": np.nan,
        "tdep_alpha1": np.nan,
        "tdep_beta1": np.nan,
        "tdep_gamma1": np.nan,
    }

    if n_params == 3:
        row.update(
            {
                "global_alpha": global_params[0],
                "global_beta": global_params[1],
                "global_gamma": global_params[2],
                "tdep_alpha0": tdep_params[0],
                "tdep_beta0": tdep_params[1],
                "tdep_gamma0": tdep_params[2],
                "tdep_alpha1": tdep_params[3],
                "tdep_beta1": tdep_params[4],
                "tdep_gamma1": tdep_params[5],
            }
        )
    else:
        row.update(
            {
                "global_alpha": global_params[0],
                "global_gamma": global_params[1],
                "tdep_alpha0": tdep_params[0],
                "tdep_gamma0": tdep_params[1],
                "tdep_alpha1": tdep_params[2],
                "tdep_gamma1": tdep_params[3],
            }
        )
    return row


def run_sigma_t_standalone():
    print("Loading data ...")
    df = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)
    df = impute_rpv_cross_section(df)
    df = impute_ss316l_composition(df)
    base_mask = (
        (df["Notch"] == "unnotched")
        & df["Creep Rupture Life (h)"].notna()
        & (df["Creep Rupture Life (h)"] > 0)
        & (df["Stress (Mpa)"] > 0)
        & df["Specimen Shape"].isin(SHAPES)
    )
    base = df[base_mask].copy()
    base["log_tR"] = np.log10(base["Creep Rupture Life (h)"])
    base["log_sigma"] = np.log10(base["Stress (Mpa)"])

    rows = []
    param_rows = []

    for dataset, cfg in DATASETS.items():
        alloys = cfg["alloys"]
        n_params = cfg["sigma_params"]
        needed = [
            "log_tR",
            "log_sigma",
            "Temperature of Creep Test (C)",
            "Thickness (mm)",
            "Aspect Ratio (L/D)",
        ]
        if n_params == 3:
            needed.append("Cross-Sectional Area")

        data = base[base["Alloy Designation"].isin(alloys)].copy()
        data = data[needed].dropna().reset_index(drop=True)

        if len(data) < 10:
            rows.append({"dataset": dataset, "n": len(data), "status": "too_small"})
            continue

        log_tR = data["log_tR"].values.astype(float)
        log_sigma = data["log_sigma"].values.astype(float)
        temperature = data["Temperature of Creep Test (C)"].values.astype(float)
        t_norm = (temperature - float(temperature.mean())) / max(float(temperature.std()), 1e-9)

        thickness = data["Thickness (mm)"].values.astype(float)
        aspect_ratio = data["Aspect Ratio (L/D)"].values.astype(float)
        log_b = np.log10(thickness / float(np.median(thickness)))
        log_ld = np.log10(aspect_ratio / float(np.median(aspect_ratio)))

        if n_params == 3:
            area = data["Cross-Sectional Area"].values.astype(float)
            log_area = np.log10(area / float(np.median(area)))
        else:
            log_area = np.zeros_like(log_b)

        creep_r2, creep_rmse, creep_w2x, n_eval = evaluate_corrected_stress(
            log_sigma, log_tR, temperature
        )

        global_params = optimize_global(
            n_params, log_sigma, log_tR, temperature, log_b, log_area, log_ld
        )
        log_sigma_global = apply_global(
            global_params, n_params, log_sigma, log_b, log_area, log_ld
        )
        global_r2, global_rmse, global_w2x, _ = evaluate_corrected_stress(
            log_sigma_global, log_tR, temperature
        )

        tdep_params = optimize_tdependent(
            n_params,
            log_sigma,
            log_tR,
            temperature,
            t_norm,
            log_b,
            log_area,
            log_ld,
            global_params,
        )
        log_sigma_tdep = apply_tdependent(
            tdep_params, n_params, log_sigma, t_norm, log_b, log_area, log_ld
        )
        tdep_r2, tdep_rmse, tdep_w2x, _ = evaluate_corrected_stress(
            log_sigma_tdep, log_tR, temperature
        )

        rows.append(
            {
                "dataset": dataset,
                "alloys": ",".join(alloys),
                "n": len(data),
                "n_eval": n_eval,
                "sigma_params": n_params,
                "status": "ok",
                "creep_law_r2": creep_r2,
                "extended_global_sigma_star_r2": global_r2,
                "tdependent_sigma_star_r2": tdep_r2,
                "delta_tdep_vs_creep": tdep_r2 - creep_r2,
                "delta_tdep_vs_global": tdep_r2 - global_r2,
                "tdep_rmse": tdep_rmse,
                "tdep_within_2x_pct": tdep_w2x,
                "creep_law_rmse": creep_rmse,
                "global_rmse": global_rmse,
                "creep_law_within_2x_pct": creep_w2x,
                "global_within_2x_pct": global_w2x,
            }
        )
        param_rows.append(parameter_row(dataset, n_params, global_params, tdep_params))

        print(
            f"{dataset:<14} n={len(data):4d} "
            f"Creep={creep_r2:.4f} Global={global_r2:.4f} "
            f"Tdep={tdep_r2:.4f} DeltaGlobal={tdep_r2 - global_r2:+.4f}"
        )

    summary = pd.DataFrame(rows)
    params = pd.DataFrame(param_rows)
    return summary, params


In [2]:
# Run Step 1
sigma_t_summary, sigma_t_params = run_sigma_t_standalone()


Loading data ...
  [RPV] imputed Cross-Sectional Area: 15 cylindrical (formula), 47 flat (median)
  [SS316L] imputed composition for 15 records using SS316L per-column medians
SS316 Family   n=1280 Creep=0.8081 Global=0.8496 Tdep=0.8630 DeltaGlobal=+0.0135
SS304 Family   n= 923 Creep=0.6558 Global=0.7211 Tdep=0.7243 DeltaGlobal=+0.0032
RPV Family     n= 135 Creep=0.9101 Global=0.9197 Tdep=0.9207 DeltaGlobal=+0.0010
SS316H         n=1066 Creep=0.8489 Global=0.8920 Tdep=0.9001 DeltaGlobal=+0.0081
SS304H         n= 834 Creep=0.7102 Global=0.7256 Tdep=0.7273 DeltaGlobal=+0.0018
SS316L         n= 120 Creep=0.7453 Global=0.7974 Tdep=0.8049 DeltaGlobal=+0.0074
SS304J         n=  69 Creep=0.9634 Global=0.9655 Tdep=0.9656 DeltaGlobal=+0.0001
SS316FR        n=  61 Creep=0.9070 Global=0.9070 Tdep=0.9070 DeltaGlobal=+0.0000
A709           n=  52 Creep=0.6734 Global=0.7967 Tdep=0.7968 DeltaGlobal=+0.0001
IN-617         n=  43 Creep=0.8877 Global=0.8877 Tdep=0.8877 DeltaGlobal=+0.0000
SS316LN       

## Step 2: Add LMP and single-parameter fractal baselines

This step computes Larson--Miller parameter and basic fractal-stress baselines and combines them with the Step 1 results.


In [3]:
"""
Complete formula-level comparison.

This step combines:
    1. Larson-Miller parameter (LMP)
    2. Pure/Norton creep law
    3. Basic fractal stress using thickness only
    4. Global extended geometry-corrected stress sigma*
    5. Temperature-dependent geometry-corrected stress sigma*_T

The global and temperature-dependent results are taken from the Step 1
results.
LMP and basic fractal stress are computed here using the same data.
All results are in-sample formula-level fits, not ML and not cross-validation.
"""

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import mean_squared_error, r2_score


DATA_PATH = os.path.join("..", "Dataset", "Creep_Properties_Dataset.xlsx")

COMP_FEATURES = [
    "C, wt.%", "Si, wt.%", "Mn, wt.%", "P, wt.%", "S, wt.%", "Ni, wt.%",
    "Cr, wt.%", "Mo, wt.%", "Al, wt.%", "N, wt.%", "Ti, wt.%", "Fe, wt.%",
    "Nb, wt.%", "B, wt.%", "Cu, wt.%", "Co, wt.%",
]

RPV_ALLOYS = {"SA533B", "SA508", "A508/A533"}

DATASETS = {
    "SS316 Family": ["SS316H", "SS316L", "SS316LN", "SS316FR"],
    "SS304 Family": ["SS304H", "SS304J", "SS304L"],
    "RPV Family": ["SA533B", "SA508", "A508/A533"],
    "SS316H": ["SS316H"],
    "SS304H": ["SS304H"],
    "SS316L": ["SS316L"],
    "SS304J": ["SS304J"],
    "SS316FR": ["SS316FR"],
    "A709": ["A709"],
    "IN-617": ["IN-617"],
    "SS316LN": ["SS316LN"],
}


def impute_rpv_cross_section(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same RPV Cross-Sectional Area handling used in all tables."""
    df = df.copy()
    rpv_mask = df["Alloy Designation"].isin(RPV_ALLOYS)
    missing = rpv_mask & df["Cross-Sectional Area"].isna()

    cyl = missing & (df["Specimen Shape"] == "Cylindrical")
    d = df.loc[cyl, "Thickness (mm)"]
    df.loc[cyl, "Cross-Sectional Area"] = np.pi * (d / 2) ** 2
    n_cyl = int(cyl.sum())

    flat = missing & (df["Specimen Shape"] == "Flat")
    rpv_median_overall = df.loc[
        rpv_mask & df["Cross-Sectional Area"].notna(), "Cross-Sectional Area"
    ].median()
    n_flat = 0
    for alloy in RPV_ALLOYS:
        alloy_flat = flat & (df["Alloy Designation"] == alloy)
        if not alloy_flat.any():
            continue
        alloy_median = df.loc[
            rpv_mask
            & (df["Alloy Designation"] == alloy)
            & df["Cross-Sectional Area"].notna(),
            "Cross-Sectional Area",
        ].median()
        fill_val = alloy_median if not np.isnan(alloy_median) else rpv_median_overall
        df.loc[alloy_flat, "Cross-Sectional Area"] = fill_val
        n_flat += int(alloy_flat.sum())

    print(
        f"  [RPV fix] imputed Cross-Sectional Area: "
        f"{n_cyl} cylindrical (formula), {n_flat} flat (median)"
    )
    return df


def impute_ss316l_composition(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same SS316L composition median fill used in all tables."""
    df = df.copy()
    mask = df["Alloy Designation"] == "SS316L"
    comp_missing = mask & df[COMP_FEATURES[0]].isna()
    if not comp_missing.any():
        return df

    medians = df.loc[mask & ~comp_missing, COMP_FEATURES].median()
    for col in COMP_FEATURES:
        df.loc[comp_missing, col] = medians[col]
    print(
        f"  [SS316L fix] imputed composition for {int(comp_missing.sum())} records "
        f"using SS316L per-column medians"
    )
    return df


def metrics(y_true, y_pred):
    return (
        float(r2_score(y_true, y_pred)),
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(np.mean(np.abs(y_true - y_pred) < np.log10(2.0)) * 100.0),
    )


def per_temperature_prediction(log_sigma_like, log_t_r, temperature):
    pred = np.full(len(log_t_r), np.nan)
    for tv in np.unique(temperature):
        mask = temperature == tv
        if int(mask.sum()) < 3:
            continue
        if float(np.std(log_sigma_like[mask])) < 1e-12:
            pred[mask] = float(np.mean(log_t_r[mask]))
        else:
            coef = np.polyfit(log_sigma_like[mask], log_t_r[mask], 1)
            pred[mask] = np.polyval(coef, log_sigma_like[mask])
    valid = np.isfinite(pred)
    if int(valid.sum()) < 2:
        return None
    r2, rmse, w2 = metrics(log_t_r[valid], pred[valid])
    return r2, rmse, w2, int(valid.sum())


def fit_lmp(data):
    needed = ["log_tR", "log_sigma", "T_kelvin"]
    sub = data[needed].dropna().reset_index(drop=True)
    if len(sub) < 5:
        return np.nan, np.nan, np.nan, len(sub), np.nan

    y = sub["log_tR"].values.astype(float)
    log_sigma = sub["log_sigma"].values.astype(float)
    t_kelvin = sub["T_kelvin"].values.astype(float)

    best = (-np.inf, np.nan, None)
    for c_lmp in np.linspace(5, 40, 71):
        lmp = t_kelvin * (c_lmp + y)
        try:
            coef = np.polyfit(log_sigma, lmp, 2)
        except np.linalg.LinAlgError:
            continue
        lmp_pred = np.polyval(coef, log_sigma)
        pred = lmp_pred / t_kelvin - c_lmp
        r2 = float(r2_score(y, pred))
        if r2 > best[0]:
            best = (r2, c_lmp, pred)

    if best[2] is None:
        return np.nan, np.nan, np.nan, len(sub), np.nan
    r2, rmse, w2 = metrics(y, best[2])
    return r2, rmse, w2, len(sub), best[1]


def fit_basic_fractal(data):
    needed = ["log_tR", "log_sigma", "Temperature of Creep Test (C)", "Thickness (mm)"]
    sub = data[needed].dropna().reset_index(drop=True)
    sub = sub[sub["Thickness (mm)"] > 0].reset_index(drop=True)
    if len(sub) < 10:
        return np.nan, np.nan, np.nan, len(sub), np.nan, 0

    y = sub["log_tR"].values.astype(float)
    log_sigma = sub["log_sigma"].values.astype(float)
    temperature = sub["Temperature of Creep Test (C)"].values.astype(float)
    thickness = sub["Thickness (mm)"].values.astype(float)
    log_b = np.log10(thickness / float(np.median(thickness)))

    def objective(alpha):
        out = per_temperature_prediction(log_sigma + float(alpha) * log_b, y, temperature)
        if out is None:
            return 1e9
        return -out[0]

    result = minimize_scalar(objective, bounds=(-0.5, 0.5), method="bounded")
    alpha = float(result.x)
    out = per_temperature_prediction(log_sigma + alpha * log_b, y, temperature)
    if out is None:
        return np.nan, np.nan, np.nan, len(sub), alpha, 0
    r2, rmse, w2, n_eval = out
    return r2, rmse, w2, len(sub), alpha, n_eval


def run_complete_formula_comparison(sigma_t):
    print("Loading data ...")
    df = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)
    print("Applying data fixes ...")
    df = impute_rpv_cross_section(df)
    df = impute_ss316l_composition(df)
    base_mask = (
        (df["Notch"] == "unnotched")
        & df["Alloy Designation"].notna()
        & df["Creep Rupture Life (h)"].notna()
        & (df["Creep Rupture Life (h)"] > 0)
        & df["Stress (Mpa)"].notna()
        & (df["Stress (Mpa)"] > 0)
        & df["Specimen Shape"].isin(["Flat", "Cylindrical"])
    )
    base = df[base_mask].copy().reset_index(drop=True)
    base["log_tR"] = np.log10(base["Creep Rupture Life (h)"])
    base["log_sigma"] = np.log10(base["Stress (Mpa)"])
    base["T_kelvin"] = base["Temperature of Creep Test (C)"] + 273.15

    rows = []
    for dataset, alloys in DATASETS.items():
        data = base[base["Alloy Designation"].isin(alloys)].copy().reset_index(drop=True)
        lmp_r2, lmp_rmse, lmp_w2, lmp_n, lmp_c = fit_lmp(data)
        basic_r2, basic_rmse, basic_w2, basic_n, basic_alpha, basic_n_eval = fit_basic_fractal(data)

        existing = sigma_t[sigma_t["dataset"] == dataset]
        if existing.empty:
            existing_row = {}
        else:
            existing_row = existing.iloc[0].to_dict()

        rows.append(
            {
                "dataset": dataset,
                "alloys": ",".join(alloys),
                "n_total": int(len(data)),
                "lmp_r2": lmp_r2,
                "lmp_rmse": lmp_rmse,
                "lmp_within_2x_pct": lmp_w2,
                "lmp_n": int(lmp_n),
                "lmp_C": lmp_c,
                "pure_creep_law_r2": existing_row.get("creep_law_r2", np.nan),
                "pure_creep_law_rmse": existing_row.get("creep_law_rmse", np.nan),
                "pure_creep_law_within_2x_pct": existing_row.get("creep_law_within_2x_pct", np.nan),
                "basic_fractal_r2": basic_r2,
                "basic_fractal_rmse": basic_rmse,
                "basic_fractal_within_2x_pct": basic_w2,
                "basic_fractal_n": int(basic_n),
                "basic_fractal_n_eval": int(basic_n_eval),
                "basic_fractal_alpha": basic_alpha,
                "global_sigma_star_r2": existing_row.get("extended_global_sigma_star_r2", np.nan),
                "global_sigma_star_rmse": existing_row.get("global_rmse", np.nan),
                "global_sigma_star_within_2x_pct": existing_row.get("global_within_2x_pct", np.nan),
                "tdependent_sigma_star_r2": existing_row.get("tdependent_sigma_star_r2", np.nan),
                "tdependent_sigma_star_rmse": existing_row.get("tdep_rmse", np.nan),
                "tdependent_sigma_star_within_2x_pct": existing_row.get("tdep_within_2x_pct", np.nan),
            }
        )

    out = pd.DataFrame(rows)
    print(out[[
        "dataset",
        "n_total",
        "lmp_r2",
        "pure_creep_law_r2",
        "basic_fractal_r2",
        "global_sigma_star_r2",
        "tdependent_sigma_star_r2",
    ]].to_string(index=False))
    return out


In [4]:
# Run Step 2
results = run_complete_formula_comparison(sigma_t_summary)


Loading data ...
Applying data fixes ...
  [RPV fix] imputed Cross-Sectional Area: 15 cylindrical (formula), 47 flat (median)
  [SS316L fix] imputed composition for 15 records using SS316L per-column medians
     dataset  n_total   lmp_r2  pure_creep_law_r2  basic_fractal_r2  global_sigma_star_r2  tdependent_sigma_star_r2
SS316 Family     1280 0.658387           0.808131          0.811385              0.849563                  0.863035
SS304 Family      923 0.332019           0.655812          0.718260              0.721147                  0.724312
  RPV Family      135 0.478655           0.910082          0.919523              0.919652                  0.920682
      SS316H     1066 0.675132           0.848927          0.871750              0.891977                  0.900078
      SS304H      834 0.364690           0.710237          0.724619              0.725558                  0.727327
      SS316L      120 0.718417           0.745290          0.759222              0.797443       

## Final Table 1


In [5]:
COLUMN_MAP = {
    "dataset": "Dataset",
    "n_total": "n",
    "lmp_r2": "LMP R2",
    "pure_creep_law_r2": "Pure/Norton creep law R2",
    "basic_fractal_r2": "Basic fractal R2",
    "global_sigma_star_r2": "Global sigma* R2",
    "tdependent_sigma_star_r2": "sigma*_T R2",
}
final_table = results[list(COLUMN_MAP)].rename(columns=COLUMN_MAP).reset_index(drop=True)
final_table

,Dataset,n,LMP R2,Pure/Norton creep law R2,Basic fractal R2,Global sigma* R2,sigma*_T R2
0,SS316 Family,1280,0.658387,0.808131,0.811385,0.849563,0.863035
1,SS304 Family,923,0.332019,0.655812,0.718260,0.721147,0.724312
2,RPV Family,135,0.478655,0.910082,0.919523,0.919652,0.920682
3,SS316H,1066,0.675132,0.848927,0.871750,0.891977,0.900078
4,SS304H,834,0.364690,0.710237,0.724619,0.725558,0.727327
5,SS316L,120,0.718417,0.745290,0.759222,0.797443,0.804866
6,SS304J,69,0.921484,0.963404,0.965476,0.965476,0.965612
7,SS316FR,61,0.919906,0.907011,0.907011,0.907011,0.907011
8,A709,52,0.636387,0.673373,0.796620,0.796733,0.796786
9,IN-617,43,0.841871,0.887698,0.887698,0.887698,0.887698
